# 🧑‍🍳 Neosantara AI Cookbook: RAG with ChromaDB

This notebook demonstrates how to build a Retrieval-Augmented Generation (RAG) system using Neosantara AI embeddings and chat completions with ChromaDB.

In [ ]:
!pip install openai chromadb

try:
    from google.colab import userdata
    import os
    os.environ["NEOSANTARA_API_KEY"] = userdata.get('NEOSANTARA_API_KEY')
except ImportError:
    import os
    # Set your key manually if not in Colab
    # os.environ["NEOSANTARA_API_KEY"] = "your_nsk_key"

In [ ]:
from openai import OpenAI
import os
import chromadb

client = OpenAI(
    api_key=os.getenv("NEOSANTARA_API_KEY"),
    base_url="https://api.neosantara.xyz/v1"
)

In [ ]:
def get_embedding(text):
    response = client.embeddings.create(
        input=text,
        model="nusa-embedding-0001"
    )
    return response.data[0].embedding

In [ ]:
db = chromadb.Client()
collection = db.create_collection(name="neosantara_docs")

docs = [
    "Neosantara AI provides Indonesian-optimized LLMs.",
    "The gateway supports OpenAI and Anthropic compatible APIs.",
    "You can use nusa-embedding-0001 for vector search."
]

for i, doc in enumerate(docs):
    collection.add(
        documents=[doc],
        embeddings=[get_embedding(doc)],
        ids=[f"id{i}"]
    )

In [ ]:
query = "What embeddings are available?"
results = collection.query(query_embeddings=[get_embedding(query)], n_results=1)
context = results['documents'][0][0]

response = client.chat.completions.create(
    model="nusantara-base",
    messages=[
        {"role": "system", "content": "Use the context to answer."},
        {"role": "user", "content": f"Context: {context}\nQuestion: {query}"}
    ]
)

print(response.choices[0].message.content)